# LLM GMC — Name That LLM baseline

This competition-linked notebook computes a reproducible local validation score. It does not submit predictions to the leaderboard.

Baseline: word + character TF-IDF features with class-balanced LinearSVC. If `prompt_id` is present, validation is grouped by prompt to reduce prompt leakage.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

DATA_DIR = Path('/kaggle/input/competitions/name-that-llm')
print('Dataset directory exists:', DATA_DIR.exists())
print('Dataset files:', sorted(p.name for p in DATA_DIR.glob('*')) if DATA_DIR.exists() else [])
assert (DATA_DIR / 'train.csv').exists(), f'train.csv not found under {DATA_DIR}'
train = pd.read_csv(DATA_DIR / 'train.csv')
print('Train shape:', train.shape)
print('Columns:', train.columns.tolist())
assert {'text', 'model'}.issubset(train.columns), 'Expected text and model columns'
train['text'] = train['text'].fillna('').astype(str)
train['model'] = train['model'].astype(str)
print('Model classes:', train['model'].nunique())
display(train['model'].value_counts())

In [ ]:
if 'prompt_id' in train.columns and train['prompt_id'].notna().any():
    groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str)
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    train_idx, valid_idx = next(splitter.split(train, groups=groups))
    split_method = 'GroupShuffleSplit by prompt_id'
else:
    train_idx, valid_idx = train_test_split(range(len(train)), test_size=0.20, random_state=42, stratify=train['model'])
    split_method = 'stratified random split (prompt_id unavailable)'

train_part = train.iloc[list(train_idx)].copy()
valid_part = train.iloc[list(valid_idx)].copy()
print('Validation method:', split_method)
print('Training rows:', len(train_part))
print('Validation rows:', len(valid_part))
print('Validation classes:', valid_part['model'].nunique())

In [ ]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

# Stylometry feature-group ablation on identical prompt-grouped folds.
texts = train['text'].fillna('').astype(str).to_numpy()
labels = train['model'].astype(str).to_numpy()
groups = train['prompt_id'].fillna('__missing_prompt_id__').astype(str).to_numpy()

function_words = ['the','a','an','and','but','or','if','to','of','in','on','for','with','as','is','are','was','be','that','this','it','I','we','you','can','would','should']
punct_chars = '.,;:!?()[]{}"\'/-'

def feature_groups(text):
    words = re.findall(r"\b[\w']+\b", text.lower())
    n_words, n_chars = max(len(words),1), max(len(text),1)
    lengths = np.asarray([len(w) for w in words], dtype=float)
    structure = [
        np.log1p(len(text)), np.log1p(len(words)), np.log1p(max(len(re.findall(r'[.!?]+', text)),1)),
        lengths.mean() if len(lengths) else 0., lengths.std() if len(lengths) else 0.,
        len(set(words))/n_words,
    ]
    punctuation = [text.count(c)/n_chars for c in punct_chars]
    formatting = [text.count('\\n')/n_chars, text.count('\\n\\n')/n_chars, text.count('#')/n_chars, text.count('*')/n_chars, text.count('```')/n_chars]
    function = [words.count(w)/n_words for w in function_words]
    caps_digits = [sum(c.isdigit() for c in text)/n_chars, sum(c.isupper() for c in text)/n_chars]
    return {
        'length_structure': structure,
        'punctuation': punctuation,
        'formatting': formatting,
        'function_words': function,
        'capitalization_digits': caps_digits,
    }

raw = [feature_groups(x) for x in texts]
style_groups = {k: np.asarray([r[k] for r in raw], dtype=np.float32) for k in raw[0]}
configs = {
    'word_char': [],
    'word_char_length_structure': ['length_structure'],
    'word_char_punctuation': ['punctuation'],
    'word_char_formatting': ['formatting'],
    'word_char_function_words': ['function_words'],
    'word_char_capitalization_digits': ['capitalization_digits'],
    'word_char_all_style': list(style_groups.keys()),
}

def make_text_features():
    return FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1,3), min_df=2, sublinear_tf=True, max_features=120000)),
        ('char', TfidfVectorizer(analyzer='char', ngram_range=(2,5), min_df=2, sublinear_tf=True, max_features=120000)),
    ])

fold_indices=list(GroupKFold(n_splits=5).split(texts,labels,groups=groups))
results={}
for config, selected in configs.items():
    scores=[]
    for fold,(tr,va) in enumerate(fold_indices,1):
        tf=make_text_features()
        Xtr_text=tf.fit_transform(texts[tr]); Xva_text=tf.transform(texts[va])
        Xtr=Xtr_text; Xva=Xva_text
        if selected:
            tr_style=np.hstack([style_groups[k][tr] for k in selected])
            va_style=np.hstack([style_groups[k][va] for k in selected])
            scaler=StandardScaler()
            Xtr=hstack([Xtr,csr_matrix(scaler.fit_transform(tr_style))],format='csr')
            Xva=hstack([Xva,csr_matrix(scaler.transform(va_style))],format='csr')
        model=LinearSVC(C=1.0,class_weight='balanced',max_iter=1200)
        model.fit(Xtr,labels[tr])
        score=float(balanced_accuracy_score(labels[va],model.predict(Xva)))
        scores.append(score)
        print(f'{config} fold {fold}: {score:.4f}')
    results[config]={'mean_balanced_accuracy':float(np.mean(scores)),'std_balanced_accuracy':float(np.std(scores,ddof=1)),'fold_scores':scores}
    print(f'CONFIG {config}: mean={np.mean(scores):.4f}')

ranked=sorted(results.items(),key=lambda kv:kv[1]['mean_balanced_accuracy'],reverse=True)
summary={'experiment':'stylometry feature-group ablation','split_method':'5-fold GroupKFold by prompt_id','configs':results,'ranking':[{'config':k,**v} for k,v in ranked]}
print('\\nSUMMARY:',json.dumps(summary,indent=2))
Path('/kaggle/working/style_group_results.json').write_text(json.dumps(summary,indent=2))
print('Saved:', '/kaggle/working/style_group_results.json')


## Stylometry feature-group ablation

All configurations use identical word/character TF-IDF and GroupKFold folds; only the selected handcrafted feature group changes.
